In [1]:
import pandas as pd
import requests
import time
import random
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv
import json
import logging
from urllib.parse import quote
import openpyxl
from openpyxl.styles import Font, Alignment
import warnings
warnings.filterwarnings('ignore')

class NaverImageAPISearcher:
    def __init__(self, search_year=2024, search_month=6, max_total_images=8000):
        # .env 파일에서 API 키 로드
        load_dotenv()
        self.client_id = os.getenv('Client_ID')
        self.client_secret = os.getenv('Client_Secret')
        
        if not self.client_id or not self.client_secret:
            raise ValueError("네이버 API 키가 설정되지 않았습니다.")
        
        self.search_year = search_year
        self.search_month = search_month
        self.max_total_images = max_total_images
        self.current_total_count = 0
        
        self.api_url = "https://openapi.naver.com/v1/search/image"
        self.headers = {
            'X-Naver-Client-Id': self.client_id,
            'X-Naver-Client-Secret': self.client_secret
        }
        
        self.results = []
        self.collected_urls = set()
        self.menu_popularity = {}
        self.api_response_sample = None  # API 응답 샘플 저장용
        
        logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
        self.logger = logging.getLogger(__name__)
        
        self.request_delay = 0.1
        self.daily_request_count = 0
        self.max_daily_requests = 20000
        
        print(f"네이버 이미지 API 검색기 초기화 완료")
        print(f"검색 기간: {search_year}년 {search_month}월")
        print(f"목표 수집 이미지: {max_total_images}개")
    
    def load_menu_data(self, csv_file_path):
        try:
            df = pd.read_csv(csv_file_path, encoding='utf-8')
            self.logger.info(f"CSV 파일 로드 완료: {len(df)}개 행")
            
            menu_items = []
            for _, row in df.iterrows():
                detail_menus = [menu.strip() for menu in str(row['상세메뉴']).split(',') if menu.strip()]
                for detail_menu in detail_menus:
                    menu_items.append({
                        '대분류': row['대분류'],
                        '중분류': row['중분류'],
                        '소분류': row['소분류'],
                        '상세메뉴': detail_menu,
                        '시각적특징': row['시각적특징']
                    })
            
            self.logger.info(f"총 {len(menu_items)}개의 개별 메뉴 항목 생성")
            return menu_items
            
        except Exception as e:
            self.logger.error(f"CSV 파일 로드 실패: {e}")
            raise
    
    def search_images_api(self, query, start=1, display=100, sort='date'):
        try:
            if self.daily_request_count >= self.max_daily_requests:
                self.logger.warning("일일 API 요청 한도에 도달했습니다.")
                return None
            
            params = {
                'query': query,
                'start': start,
                'display': min(display, 100),
                'sort': sort,
                'filter': 'all'
            }
            
            response = requests.get(self.api_url, headers=self.headers, params=params, timeout=10)
            self.daily_request_count += 1
            
            if response.status_code == 200:
                result = response.json()
                
                # 첫 번째 API 응답 샘플 저장 (디버깅용)
                if self.api_response_sample is None and result.get('items'):
                    self.api_response_sample = result['items'][0]
                    print(f"\nAPI 응답 샘플 (첫 번째 이미지 필드들):")
                    for key, value in self.api_response_sample.items():
                        print(f"  {key}: {value}")
                    print()
                
                return result
            elif response.status_code == 429:
                print(f"API 제한 도달, 10초 대기...")
                time.sleep(10)
                return None
            else:
                self.logger.error(f"API 요청 실패: {response.status_code}")
                return None
                
        except Exception as e:
            self.logger.error(f"API 요청 중 오류: {e}")
            return None
        finally:
            time.sleep(self.request_delay)
    
    def is_valid_date_flexible(self, api_image):
        """
        더 유연한 날짜 확인 - pubDate가 없으면 일단 포함시키고,
        있으면 해당 시점인지 확인
        """
        pub_date = api_image.get('pubDate', '')
        
        # pubDate가 없으면 일단 포함 (나중에 다른 방법으로 필터링)
        if not pub_date:
            return True
        
        try:
            # 다양한 날짜 형식 시도
            date_formats = [
                '%a, %d %b %Y %H:%M:%S %z',  # 'Wed, 01 Jun 2024 12:00:00 +0900'
                '%Y-%m-%d',  # '2024-06-01'
                '%Y/%m/%d',  # '2024/06/01'
                '%Y.%m.%d',  # '2024.06.01'
            ]
            
            parsed_date = None
            for fmt in date_formats:
                try:
                    parsed_date = datetime.strptime(pub_date.strip(), fmt)
                    break
                except:
                    continue
            
            if parsed_date:
                return (parsed_date.year == self.search_year and 
                       parsed_date.month == self.search_month)
            
            # 문자열 파싱 시도
            if isinstance(pub_date, str):
                parts = pub_date.split()
                if len(parts) >= 4:
                    year_str = parts[3] if len(parts) > 3 else parts[-1]
                    month_str = parts[2] if len(parts) > 2 else ''
                    
                    try:
                        year = int(year_str)
                        month_map = {
                            'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
                            'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
                            'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
                        }
                        month = month_map.get(month_str, 0)
                        
                        return year == self.search_year and month == self.search_month
                    except:
                        pass
            
        except Exception as e:
            pass
        
        # 파싱 실패시 일단 포함
        return True
    
    def calculate_menu_popularity_simple(self, menu_items):
        """
        간단한 인기도 측정 - 전체 검색 결과 수 기반
        """
        print(f"메뉴별 검색 결과 수 기반 인기도 측정 중...")
        popularity_scores = {}
        
        for i, menu_item in enumerate(menu_items):
            menu_name = menu_item['상세메뉴']
            print(f"[{i+1}/{len(menu_items)}] '{menu_name}' 인기도 측정")
            
            result = self.search_images_api(menu_name, start=1, display=1)
            
            if result and 'total' in result:
                total_count = result['total']
                popularity_scores[menu_name] = total_count
                print(f"  -> 전체 검색 결과: {total_count}개")
            else:
                popularity_scores[menu_name] = 1  # 최소값 1
                print(f"  -> 검색 결과 없음 (기본값 1)")
            
            time.sleep(random.uniform(0.1, 0.2))
        
        self.menu_popularity = popularity_scores
        
        # 인기도 순으로 정렬
        sorted_popularity = sorted(popularity_scores.items(), key=lambda x: x[1], reverse=True)
        
        print(f"\n메뉴 인기도 TOP 15")
        for i, (menu, count) in enumerate(sorted_popularity[:15]):
            print(f"{i+1:2d}. {menu:<15}: {count:4d}개")
        
        return popularity_scores
    
    def calculate_menu_quotas(self, menu_items, popularity_scores):
        total_popularity = sum(popularity_scores.values())
        
        if total_popularity == 0:
            print("인기도 합계가 0이므로 균등 분배합니다.")
            quota_per_menu = self.max_total_images // len(menu_items)
            return {item['상세메뉴']: quota_per_menu for item in menu_items}
        
        quotas = {}
        
        for menu_item in menu_items:
            menu_name = menu_item['상세메뉴']
            popularity = popularity_scores.get(menu_name, 1)
            
            ratio = popularity / total_popularity
            quota = max(1, int(self.max_total_images * ratio))  # 최소 1개
            
            quotas[menu_name] = quota
        
        # 할당량 조정 (총합이 목표치와 맞지 않을 수 있음)
        current_total = sum(quotas.values())
        if current_total != self.max_total_images:
            # 비율 조정
            adjustment_ratio = self.max_total_images / current_total
            for menu_name in quotas:
                quotas[menu_name] = max(1, int(quotas[menu_name] * adjustment_ratio))
        
        print(f"\n메뉴별 수집 할당량 TOP 15")
        sorted_quotas = sorted(quotas.items(), key=lambda x: x[1], reverse=True)
        allocated_total = 0
        for i, (menu, quota) in enumerate(sorted_quotas[:15]):
            print(f"{i+1:2d}. {menu:<15}: {quota:4d}개")
            allocated_total += quota
        
        print(f"\n총 할당량: {allocated_total}개")
        
        return quotas
    
    def search_menu_images(self, menu_item, target_quota):
        menu_name = menu_item['상세메뉴']
        collected_images = []
        
        if target_quota <= 0:
            return []
        
        search_keywords = [
            menu_name,
            f"{menu_name} 음식",
            f"{menu_name} 요리",
            f"음식 {menu_name}",
            f"한국음식 {menu_name}",
            f"{menu_name} 이미지"
        ]
        
        print(f"  검색 키워드: {len(search_keywords)}개")
        
        for keyword in search_keywords:
            if len(collected_images) >= target_quota:
                break
            
            for sort_method in ['date', 'sim']:
                if len(collected_images) >= target_quota:
                    break
                
                start = 1
                max_pages = 5  # 키워드당 최대 5페이지
                
                for page in range(max_pages):
                    if len(collected_images) >= target_quota:
                        break
                    
                    result = self.search_images_api(keyword, start=start, display=100, sort=sort_method)
                    
                    if not result or 'items' not in result:
                        break
                    
                    images = result['items']
                    if not images:
                        break
                    
                    valid_images = []
                    for img in images:
                        img_url = img.get('link', '')
                        
                        if (img_url and 
                            img_url not in self.collected_urls and
                            self.is_valid_date_flexible(img)):
                            
                            valid_images.append(img)
                            self.collected_urls.add(img_url)
                    
                    if valid_images:
                        collected_images.extend(valid_images)
                        print(f"    '{keyword}' ({sort_method}, p{page+1}): +{len(valid_images)}개")
                    
                    start += len(images)
                    
                    if len(images) < 100:
                        break
                
                time.sleep(random.uniform(0.1, 0.2))
        
        final_images = collected_images[:target_quota]
        
        processed_images = []
        for img in final_images:
            processed_img = self.process_image_data(img, menu_item, f"검색:{menu_name}")
            processed_images.append(processed_img)
        
        return processed_images
    
    def process_image_data(self, api_image, menu_item, search_keyword):
        # API 응답에서 실제 사용 가능한 필드만 추출
        return {
            'image_url': api_image.get('link', ''),
            'thumbnail_url': api_image.get('thumbnail', ''),
            'title': api_image.get('title', '').replace('<b>', '').replace('</b>', ''),
            'size_height': api_image.get('sizeheight', ''),
            'size_width': api_image.get('sizewidth', ''),
            'pub_date': api_image.get('pubDate', ''),
            # 추가 필드들 (있다면)
            'display_sitename': api_image.get('displaySitename', ''),
            '대분류': menu_item['대분류'],
            '중분류': menu_item['중분류'],
            '소분류': menu_item['소분류'],
            '상세메뉴': menu_item['상세메뉴'],
            '시각적특징': menu_item['시각적특징'],
            '업로드시기': f"{self.search_year}-{self.search_month:02d}",
            '검색키워드': search_keyword,
            'collected_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
    
    def process_all_menus(self, csv_file_path):
        menu_items = self.load_menu_data(csv_file_path)
        
        # 간단한 인기도 측정 (전체 검색 결과 수 기반)
        popularity_scores = self.calculate_menu_popularity_simple(menu_items)
        menu_quotas = self.calculate_menu_quotas(menu_items, popularity_scores)
        
        print(f"\n메뉴별 이미지 수집 시작")
        processed_count = 0
        
        # 할당량이 있는 메뉴만 처리
        valid_menus = [(item, menu_quotas.get(item['상세메뉴'], 0)) 
                       for item in menu_items if menu_quotas.get(item['상세메뉴'], 0) > 0]
        
        print(f"수집 대상 메뉴: {len(valid_menus)}개")
        
        for i, (menu_item, target_quota) in enumerate(valid_menus):
            if self.current_total_count >= self.max_total_images:
                print(f"\n목표 수집량({self.max_total_images}개)에 도달하여 중단합니다.")
                break
            
            if self.daily_request_count >= self.max_daily_requests:
                print(f"\nAPI 일일 한도에 도달하여 중단합니다.")
                break
            
            menu_name = menu_item['상세메뉴']
            
            print(f"\n[{i+1}/{len(valid_menus)}] '{menu_name}' 수집 (할당량: {target_quota}개)")
            
            images = self.search_menu_images(menu_item, target_quota)
            
            if images:
                self.results.extend(images)
                self.current_total_count += len(images)
                
                success_rate = (len(images) / target_quota * 100) if target_quota > 0 else 0
                print(f"  -> {len(images)}개 수집 완료 (달성률: {success_rate:.1f}%)")
                print(f"  -> 총 누적: {self.current_total_count}개")
            else:
                print(f"  -> 수집된 이미지 없음")
            
            processed_count += 1
            
            if processed_count % 10 == 0:
                progress = (processed_count / len(valid_menus)) * 100
                overall_success = (self.current_total_count / self.max_total_images * 100)
                print(f"\n진행률: {progress:.1f}% ({processed_count}/{len(valid_menus)})")
                print(f"전체 달성률: {overall_success:.1f}% ({self.current_total_count}/{self.max_total_images})")
                print(f"API 요청: {self.daily_request_count}회")
            
            time.sleep(random.uniform(0.3, 0.7))
        
        print(f"\n전체 처리 완료!")
        print(f"최종 수집 이미지: {self.current_total_count}개")
        print(f"목표 달성률: {(self.current_total_count/self.max_total_images)*100:.1f}%")
        print(f"처리된 메뉴: {processed_count}개")
        print(f"API 요청 횟수: {self.daily_request_count}회")
        
        # API 응답 샘플 출력
        if self.api_response_sample:
            print(f"\n수집된 API 응답 필드 확인:")
            for key, value in self.api_response_sample.items():
                print(f"  {key}: {str(value)[:100]}...")
    
    def save_to_excel(self, output_file="naver_image_flexible.xlsx"):
        if not self.results:
            print("저장할 결과가 없습니다.")
            return
        
        try:
            df = pd.DataFrame(self.results)
            
            columns_order = [
                '상세메뉴', '대분류', '중분류', '소분류', '시각적특징',
                'image_url', 'thumbnail_url', 'title', 
                'size_width', 'size_height', 'pub_date', 'display_sitename',
                '업로드시기', '검색키워드', 'collected_at'
            ]
            
            available_columns = [col for col in columns_order if col in df.columns]
            df = df[available_columns]
            
            original_count = len(df)
            df = df.drop_duplicates(subset=['image_url'], keep='first')
            removed_count = original_count - len(df)
            
            if removed_count > 0:
                print(f"중복 제거: {removed_count}개 (최종: {len(df)}개)")
            
            with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
                df.to_excel(writer, sheet_name='검색결과', index=False)
                
                worksheet = writer.sheets['검색결과']
                
                header_font = Font(bold=True)
                header_alignment = Alignment(horizontal='center')
                
                for cell in worksheet[1]:
                    cell.font = header_font
                    cell.alignment = header_alignment
                
                for column in worksheet.columns:
                    max_length = 0
                    column_letter = column[0].column_letter
                    
                    for cell in column:
                        try:
                            if len(str(cell.value)) > max_length:
                                max_length = len(str(cell.value))
                        except:
                            pass
                    
                    adjusted_width = min(max_length + 2, 50)
                    worksheet.column_dimensions[column_letter].width = adjusted_width
            
            self.save_statistics(output_file, df)
            
            print(f"\n결과 저장 완료: {output_file}")
            print(f"최종 저장: {len(df)}개 이미지 정보")
            
        except Exception as e:
            self.logger.error(f"엑셀 저장 실패: {e}")
            
            try:
                df.to_csv(output_file.replace('.xlsx', '.csv'), encoding='utf-8-sig', index=False)
                print(f"백업 CSV 파일로 저장: {output_file.replace('.xlsx', '.csv')}")
            except Exception as csv_error:
                self.logger.error(f"CSV 백업 저장도 실패: {csv_error}")
    
    def save_statistics(self, excel_file, df):
        try:
            menu_stats = df.groupby('상세메뉴').size().reset_index(name='실제수집량')
            menu_stats = menu_stats.sort_values('실제수집량', ascending=False)
            
            category_stats = df.groupby(['대분류', '중분류']).size().reset_index(name='이미지수')
            category_stats = category_stats.sort_values('이미지수', ascending=False)
            
            # 인기도 순위
            popularity_df = pd.DataFrame(list(self.menu_popularity.items()), 
                                       columns=['메뉴명', '인기도점수'])
            popularity_df = popularity_df.sort_values('인기도점수', ascending=False)
            
            with pd.ExcelWriter(excel_file, mode='a', engine='openpyxl') as writer:
                menu_stats.to_excel(writer, sheet_name='메뉴별수집량', index=False)
                popularity_df.to_excel(writer, sheet_name='인기도순위', index=False)
                category_stats.to_excel(writer, sheet_name='분류별통계', index=False)
                
                summary_data = {
                    '항목': [
                        '최종 수집 이미지',
                        '목표 달성률',
                        '처리된 메뉴 수',
                        '평균 메뉴당 수집',
                        '검색 기간',
                        '수집 완료 시각',
                        'API 총 요청',
                        '인기도 측정 방식'
                    ],
                    '값': [
                        f"{len(df)}개",
                        f"{(len(df) / self.max_total_images * 100):.1f}%",
                        f"{df['상세메뉴'].nunique()}개",
                        f"{len(df) / df['상세메뉴'].nunique():.1f}개" if df['상세메뉴'].nunique() > 0 else "0개",
                        f"{self.search_year}년 {self.search_month}월",
                        datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                        f"{self.daily_request_count}회",
                        "전체 검색 결과 수 기반"
                    ]
                }
                
                summary_df = pd.DataFrame(summary_data)
                summary_df.to_excel(writer, sheet_name='수집요약', index=False)
            
        except Exception as e:
            self.logger.error(f"통계 저장 실패: {e}")


def main():
    CSV_FILE_PATH = "식당대12중53소132상세메뉴379분류.csv"
    SEARCH_YEAR = 2025
    SEARCH_MONTH = 6
    MAX_IMAGES = 8000
    OUTPUT_FILE = f"naver_image_flexible_{SEARCH_YEAR}{SEARCH_MONTH:02d}.xlsx"
    
    searcher = None
    
    try:
        print("네이버 이미지 유연한 수집 시스템")
        
        searcher = NaverImageAPISearcher(
            search_year=SEARCH_YEAR,
            search_month=SEARCH_MONTH,
            max_total_images=MAX_IMAGES
        )
        
        if not os.path.exists(CSV_FILE_PATH):
            print(f"오류: CSV 파일을 찾을 수 없습니다 - {CSV_FILE_PATH}")
            return
        
        if not os.path.exists('.env'):
            print("\n.env 파일이 필요합니다:")
            print("NAVER_CLIENT_ID=your_client_id")
            print("NAVER_CLIENT_SECRET=your_client_secret")
            return
        
        print(f"\n유연한 방식으로 이미지 수집 시작...")
        
        searcher.process_all_menus(CSV_FILE_PATH)
        searcher.save_to_excel(OUTPUT_FILE)
        
        print("\n작업 완료!")
        
    except ValueError as ve:
        print(f"\n설정 오류: {ve}")
        
    except KeyboardInterrupt:
        print("\n사용자에 의해 중단되었습니다.")
        if searcher and searcher.results:
            print("현재까지의 결과를 저장합니다...")
            searcher.save_to_excel(f"partial_{OUTPUT_FILE}")
        
    except Exception as e:
        print(f"\n오류 발생: {e}")
        import traceback
        traceback.print_exc()
        
    finally:
        if searcher:
            print("\n시스템 종료")


if __name__ == "__main__":
    main()

2025-07-15 12:50:00,796 - INFO - CSV 파일 로드 완료: 138개 행
2025-07-15 12:50:00,816 - INFO - 총 381개의 개별 메뉴 항목 생성


네이버 이미지 유연한 수집 시스템
네이버 이미지 API 검색기 초기화 완료
검색 기간: 2024년 6월
목표 수집 이미지: 8000개

유연한 방식으로 이미지 수집 시작...
메뉴별 검색 결과 수 기반 인기도 측정 중...
[1/381] '제육볶음' 인기도 측정

API 응답 샘플 (첫 번째 이미지 필드들):
  title: 고추명가 제육볶음 소스  2kg 식당업소용 제육 양념장 배달도시락 : 고추명가
  link: http://shop1.phinf.naver.net/20250430_150/1745978683697KEVLP_JPEG/67296020691082919_1522990793.jpg
  thumbnail: https://search.pstatic.net/common/?type=b150&src=http://shop1.phinf.naver.net/20250430_150/1745978683697KEVLP_JPEG/67296020691082919_1522990793.jpg
  sizeheight: 1000
  sizewidth: 1000

  -> 전체 검색 결과: 510485개
[2/381] '매운제육볶음' 인기도 측정
  -> 전체 검색 결과: 128950개
[3/381] '두부제육볶음' 인기도 측정
  -> 전체 검색 결과: 77754개
[4/381] '된장찌개' 인기도 측정
  -> 전체 검색 결과: 669188개
[5/381] '김치찌개' 인기도 측정
  -> 전체 검색 결과: 1350662개
[6/381] '청국장찌개' 인기도 측정
  -> 전체 검색 결과: 141980개
[7/381] '콩나물무침' 인기도 측정
  -> 전체 검색 결과: 346600개
[8/381] '시금치나물' 인기도 측정
  -> 전체 검색 결과: 426867개
[9/381] '도라지무침' 인기도 측정
  -> 전체 검색 결과: 33937개
[10/381] '계란말이' 인기도 측정
  -> 전체 검색 결과: 937661개
[11/381] '계란찜' 인기도 측정
  -> 전체 검

  -> 전체 검색 결과: 127811개
[169/381] '덮밥세트' 인기도 측정
  -> 전체 검색 결과: 163127개
[170/381] '연어초밥' 인기도 측정
  -> 전체 검색 결과: 1446196개
[171/381] '참치초밥' 인기도 측정
  -> 전체 검색 결과: 1645483개
[172/381] '장어초밥' 인기도 측정
  -> 전체 검색 결과: 253836개
[173/381] '참치사시미' 인기도 측정
  -> 전체 검색 결과: 642035개
[174/381] '연어사시미' 인기도 측정
  -> 전체 검색 결과: 426953개
[175/381] '모둠사시미' 인기도 측정
  -> 전체 검색 결과: 80313개
[176/381] '연어덮밥' 인기도 측정
  -> 전체 검색 결과: 871496개
[177/381] '장어덮밥' 인기도 측정
  -> 전체 검색 결과: 356296개
[178/381] '김밥롤' 인기도 측정
  -> 전체 검색 결과: 445813개
[179/381] '캘리포니아롤' 인기도 측정
  -> 전체 검색 결과: 130468개
[180/381] '필라델피아롤' 인기도 측정
  -> 전체 검색 결과: 17695개
[181/381] '돈코츠라멘' 인기도 측정
  -> 전체 검색 결과: 1134472개
[182/381] '차슈라멘' 인기도 측정
  -> 전체 검색 결과: 625427개
[183/381] '미소라멘' 인기도 측정
  -> 전체 검색 결과: 667904개
[184/381] '쇼유라멘' 인기도 측정
  -> 전체 검색 결과: 532326개
[185/381] '시오라멘' 인기도 측정
  -> 전체 검색 결과: 633287개
[186/381] '맑은라멘' 인기도 측정
  -> 전체 검색 결과: 26660개
[187/381] '매운라멘' 인기도 측정
  -> 전체 검색 결과: 259774개
[188/381] '마제소바' 인기도 측정
  -> 전체 검색 결과: 454801개
[189/381] '매운미소' 인기도 측정
  -> 전

  -> 전체 검색 결과: 957804개
[344/381] '자몽에이드' 인기도 측정
  -> 전체 검색 결과: 461128개
[345/381] '탄산음료' 인기도 측정
  -> 전체 검색 결과: 595444개
[346/381] '치즈케이크' 인기도 측정
  -> 전체 검색 결과: 5811749개
[347/381] '초콜릿케이크' 인기도 측정
  -> 전체 검색 결과: 2593296개
[348/381] '생크림케이크' 인기도 측정
  -> 전체 검색 결과: 1542867개
[349/381] '크루아상' 인기도 측정
  -> 전체 검색 결과: 936534개
[350/381] '베이글' 인기도 측정
  -> 전체 검색 결과: 1869548개
[351/381] '머핀' 인기도 측정
  -> 전체 검색 결과: 1465474개
[352/381] '팥빙수' 인기도 측정
  -> 전체 검색 결과: 575687개
[353/381] '망고빙수' 인기도 측정
  -> 전체 검색 결과: 373203개
[354/381] '인절미빙수' 인기도 측정
  -> 전체 검색 결과: 73422개
[355/381] '마카롱' 인기도 측정
  -> 전체 검색 결과: 4209201개
[356/381] '쿠키' 인기도 측정
  -> 전체 검색 결과: 13633344개
[357/381] '마라파스타' 인기도 측정
  -> 전체 검색 결과: 176825개
[358/381] '불닭파스타' 인기도 측정
  -> 전체 검색 결과: 154743개
[359/381] '김치파스타' 인기도 측정
  -> 전체 검색 결과: 858499개
[360/381] '김치볶음밥' 인기도 측정
  -> 전체 검색 결과: 1206794개
[361/381] '한식덮밥' 인기도 측정
  -> 전체 검색 결과: 227261개
[362/381] '스시버거' 인기도 측정
  -> 전체 검색 결과: 178975개
[363/381] '라멘버거' 인기도 측정
  -> 전체 검색 결과: 124530개
[364/381] '돈까스버거' 인기도 측정



[46/381] '닭죽' 수집 (할당량: 8개)
  검색 키워드: 6개
    '닭죽' (date, p1): +58개
  -> 8개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 418개

[47/381] '삼계죽' 수집 (할당량: 1개)
  검색 키워드: 6개
    '삼계죽' (date, p1): +65개
  -> 1개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 419개

[48/381] '단호박죽' 수집 (할당량: 3개)
  검색 키워드: 6개
    '단호박죽' (date, p1): +84개
  -> 3개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 422개

[49/381] '녹두죽' 수집 (할당량: 1개)
  검색 키워드: 6개
    '녹두죽' (date, p1): +78개
  -> 1개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 423개

[50/381] '팥죽' 수집 (할당량: 5개)
  검색 키워드: 6개
    '팥죽' (date, p1): +96개
  -> 5개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 428개

진행률: 13.1% (50/381)
전체 달성률: 5.3% (428/8000)
API 요청: 431회

[51/381] '제육도시락' 수집 (할당량: 2개)
  검색 키워드: 6개
    '제육도시락' (date, p1): +90개
  -> 2개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 430개

[52/381] '불고기도시락' 수집 (할당량: 7개)
  검색 키워드: 6개
    '불고기도시락' (date, p1): +83개
  -> 7개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 437개

[53/381] '생선구이도시락' 수집 (할당량: 3개)
  검색 키워드: 6개
    '생선구이도시락' (date, p1): +99개
  -> 3개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 440개

[54/381] '국물떡볶이' 수집 (

    '우동' (date, p1): +64개
    '우동' (date, p2): +77개
    '우동' (date, p3): +39개
  -> 177개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 1912개

[113/381] '소바' 수집 (할당량: 122개)
  검색 키워드: 6개
    '소바' (date, p1): +46개
    '소바' (date, p2): +76개
  -> 122개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 2034개

[114/381] '짜장면' 수집 (할당량: 29개)
  검색 키워드: 6개
    '짜장면' (date, p1): +57개
  -> 29개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 2063개

[115/381] '삼선짜장' 수집 (할당량: 2개)
  검색 키워드: 6개
    '삼선짜장' (date, p1): +94개
  -> 2개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 2065개

[116/381] '유니짜장' 수집 (할당량: 1개)
  검색 키워드: 6개
    '유니짜장' (date, p1): +80개
  -> 1개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 2066개

[117/381] '짬뽕' 수집 (할당량: 55개)
  검색 키워드: 6개
    '짬뽕' (date, p1): +40개
    '짬뽕' (date, p2): +70개
  -> 55개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 2121개

[118/381] '삼선짬뽕' 수집 (할당량: 3개)
  검색 키워드: 6개
    '삼선짬뽕' (date, p1): +44개
  -> 3개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 2124개

[119/381] '고추짬뽕' 수집 (할당량: 3개)
  검색 키워드: 6개
    '고추짬뽕' (date, p1): +76개
  -> 3개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 2127개

[120/

    '김밥롤' (date, p1): +74개
  -> 10개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 2630개

[179/381] '캘리포니아롤' 수집 (할당량: 2개)
  검색 키워드: 6개
    '캘리포니아롤' (date, p1): +88개
  -> 2개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 2632개

[180/381] '필라델피아롤' 수집 (할당량: 1개)
  검색 키워드: 6개
    '필라델피아롤' (date, p1): +99개
  -> 1개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 2633개

진행률: 47.2% (180/381)
전체 달성률: 32.9% (2633/8000)
API 요청: 571회

[181/381] '돈코츠라멘' 수집 (할당량: 25개)
  검색 키워드: 6개
    '돈코츠라멘' (date, p1): +70개
  -> 25개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 2658개

[182/381] '차슈라멘' 수집 (할당량: 14개)
  검색 키워드: 6개
    '차슈라멘' (date, p1): +81개
  -> 14개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 2672개

[183/381] '미소라멘' 수집 (할당량: 15개)
  검색 키워드: 6개
    '미소라멘' (date, p1): +51개
  -> 15개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 2687개

[184/381] '쇼유라멘' 수집 (할당량: 12개)
  검색 키워드: 6개
    '쇼유라멘' (date, p1): +58개
  -> 12개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 2699개

[185/381] '시오라멘' 수집 (할당량: 14개)
  검색 키워드: 6개
    '시오라멘' (date, p1): +44개
  -> 14개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 2713개

[186/381] '맑은라멘' 수집 (할당량: 

    '클램차우더' (date, p1): +82개
  -> 1개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 3700개

[244/381] '치킨수프' 수집 (할당량: 7개)
  검색 키워드: 6개
    '치킨수프' (date, p1): +61개
  -> 7개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 3707개

[245/381] '바비큐립' 수집 (할당량: 1개)
  검색 키워드: 6개
    '바비큐립' (date, p1): +99개
  -> 1개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 3708개

[246/381] '풀드포크' 수집 (할당량: 2개)
  검색 키워드: 6개
    '풀드포크' (date, p1): +89개
  -> 2개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 3710개

[247/381] '브리스킷' 수집 (할당량: 3개)
  검색 키워드: 6개
    '브리스킷' (date, p1): +76개
  -> 3개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 3713개

[248/381] '타코' 수집 (할당량: 100개)
  검색 키워드: 6개
    '타코' (date, p1): +77개
    '타코' (date, p2): +93개
  -> 100개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 3813개

[249/381] '부리또' 수집 (할당량: 9개)
  검색 키워드: 6개
    '부리또' (date, p1): +91개
  -> 9개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 3822개

[250/381] '나초' 수집 (할당량: 17개)
  검색 키워드: 6개
    '나초' (date, p1): +88개
  -> 17개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 3839개

진행률: 65.6% (250/381)
전체 달성률: 48.0% (3839/8000)
API 요청: 645회

[251/381] '케사디아' 수집 (할당


[307/381] '어니언' 수집 (할당량: 16개)
  검색 키워드: 6개
    '어니언' (date, p1): +68개
  -> 16개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 5338개

[308/381] '순살치킨' 수집 (할당량: 18개)
  검색 키워드: 6개
    '순살치킨' (date, p1): +25개
  -> 18개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 5356개

[309/381] '팝콘치킨' 수집 (할당량: 4개)
  검색 키워드: 6개
    '팝콘치킨' (date, p1): +70개
  -> 4개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 5360개

[310/381] '치킨텐더' 수집 (할당량: 2개)
  검색 키워드: 6개
    '치킨텐더' (date, p1): +67개
  -> 2개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 5362개

진행률: 81.4% (310/381)
전체 달성률: 67.0% (5362/8000)
API 요청: 726회

[311/381] '윙' 수집 (할당량: 99개)
  검색 키워드: 6개
    '윙' (date, p1): +94개
    '윙' (date, p2): +100개
  -> 99개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 5461개

[312/381] '다리' 수집 (할당량: 617개)
  검색 키워드: 6개
    '다리' (date, p1): +98개
    '다리' (date, p2): +97개
    '다리' (date, p3): +62개
    '다리' (date, p4): +95개
    '다리' (date, p5): +91개
    '다리' (sim, p1): +100개
    '다리' (sim, p2): +100개
  -> 617개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 6078개

[313/381] '가슴살' 수집 (할당량: 55개)
  검색 키워드: 6개
    '가슴살' (da


[369/381] '샐러드바' 수집 (할당량: 21개)
  검색 키워드: 6개
    '샐러드바' (date, p1): +81개
  -> 21개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 7788개

[370/381] '샐러드뷔페' 수집 (할당량: 19개)
  검색 키워드: 6개
    '샐러드뷔페' (date, p1): +72개
  -> 19개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 7807개

진행률: 97.1% (370/381)
전체 달성률: 97.6% (7807/8000)
API 요청: 804회

[371/381] '호텔뷔페' 수집 (할당량: 61개)
  검색 키워드: 6개
    '호텔뷔페' (date, p1): +83개
  -> 61개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 7868개

[372/381] '브런치뷔페' 수집 (할당량: 12개)
  검색 키워드: 6개
    '브런치뷔페' (date, p1): +75개
  -> 12개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 7880개

[373/381] '비건버거' 수집 (할당량: 1개)
  검색 키워드: 6개
    '비건버거' (date, p1): +81개
  -> 1개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 7881개

[374/381] '두부스테이크' 수집 (할당량: 5개)
  검색 키워드: 6개
    '두부스테이크' (date, p1): +61개
  -> 5개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 7886개

[375/381] '템페' 수집 (할당량: 1개)
  검색 키워드: 6개
    '템페' (date, p1): +85개
  -> 1개 수집 완료 (달성률: 100.0%)
  -> 총 누적: 7887개

[376/381] '비건케이크' 수집 (할당량: 3개)
  검색 키워드: 6개
    '비건케이크' (date, p1): +91개
  -> 3개 수집 완료 (달성률: 100.0%)
  -> 총